# ADC target landscape — remote patient baseline

Run this only after `00_adc_target_landscape_remote_audit.ipynb` passes. The notebook keeps tensors in Google Drive/Colab and writes only compact patient-level tables and metrics.


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, torch
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/AI Pathology Biomarkers Project/brca')
MANIFEST = pd.read_csv(ROOT/'csvs/tcga_brca.csv')
UNI_ROOT = ROOT/'uni_features/tcga_features'
print(len(MANIFEST), MANIFEST.case_id.nunique(), UNI_ROOT)

In [ ]:
# Deterministic patient-level pooling: choose the lexicographically first slide per patient.
rows=[]
for case_id, group in MANIFEST.sort_values('slide_id').groupby('case_id'):
    slide_id = group.iloc[0].slide_id
    path = UNI_ROOT/f'{slide_id}.pt'
    if not path.exists(): continue
    tensor = torch.load(path, map_location='cpu', weights_only=False).float()
    assert tensor.ndim == 2 and tensor.shape[1] == 1024
    rows.append({'case_id':case_id, 'slide_id':slide_id, **{f'uni_{i}':float(v) for i,v in enumerate(tensor.mean(0))}})
X = pd.DataFrame(rows).set_index('case_id')
print('pooled patients:', len(X), 'features:', X.filter(like='uni_').shape[1])
X.to_parquet('/content/patient_uni_mean.parquet')

In [ ]:
# Match the six named Xena targets and evaluate out-of-fold ridge baselines.
TARGETS=['ERBB2','TACSTD2','PVRL4','FOLR1','SLC39A6','MET']
XENA_URL='https://tcga-xena-hub.s3.us-east-1.amazonaws.com/download/TCGA.BRCA.sampleMap%2FHiSeqV2_PANCAN.gz'
xena = pd.read_csv(XENA_URL, sep='\t', compression='gzip', index_col=0)
primary = xena.loc[:, [c for c in xena.columns if len(c)>=15 and c[13:15]=='01']]
primary.columns=[c[:12] for c in primary.columns]
primary=primary.T.groupby(level=0).mean().T
common=sorted(set(X.index)&set(primary.columns))
kf=KFold(n_splits=5, shuffle=True, random_state=42)
features=[c for c in X.columns if c.startswith('uni_')]
metrics=[]
for target in TARGETS:
    y=primary.loc[target, common].astype(float).to_numpy(); xx=X.loc[common, features].to_numpy()
    pred=np.full(len(common), np.nan)
    for train, test in kf.split(xx):
        model=make_pipeline(StandardScaler(), Ridge(alpha=10.0)); model.fit(xx[train], y[train]); pred[test]=model.predict(xx[test])
    metrics.append({'target':target,'n':len(common),'r2':r2_score(y,pred),'mae':mean_absolute_error(y,pred)})
display(pd.DataFrame(metrics).round(4))
pd.DataFrame(metrics).to_csv('/content/remote_baseline_metrics.csv', index=False)